In [0]:
dbutils.widgets.text("latitude", "52.52", "Latitude")
dbutils.widgets.text("longitude", "13.41", "Longitude")
dbutils.widgets.text("start_date", "2021-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2021-12-31", "End Date (YYYY-MM-DD)")
dbutils.widgets.text("hourly", "temperature_2m", "Hourly Variables")

latitude = dbutils.widgets.get("latitude")
longitude = dbutils.widgets.get("longitude")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
hourly = dbutils.widgets.get("hourly")

url = (
    f"https://archive-api.open-meteo.com/v1/era5"
    f"?latitude={latitude}"
    f"&longitude={longitude}"
    f"&start_date={start_date}"
    f"&end_date={end_date}"
    f"&hourly={hourly}"
)

print(url)


In [0]:
import requests

response = requests.get(url)
response.raise_for_status()

volume_path = "/Volumes/my/default/my"
file_name = f"meteo_date_{start_date}_{end_date}.json"
file_path = f"{volume_path}/{file_name}"

with open(file_path, "w") as f:
    f.write(response.text)

print(f"Saved API response to {file_path}")

In [0]:
from pyspark.sql import functions as F

volume_path = "/Volumes/my/default/my"
table_name = "my.default.raw_weather"

# Read each JSON file in the volume as a single record (whole file = one row)
# pathGlobFilter ensures only files starting with "meteo_date_" and ending in ".json" are read
raw_df = (
    spark.read.text(volume_path, wholetext=True, pathGlobFilter="meteo_date_*.json")
    .select(
        F.col("_metadata.file_name").alias("file_name"),
        F.col("value").alias("file_data"),
        F.current_timestamp().alias("ingestion_time"),
    )
)

# Write the raw records (file_name, file_data, ingestion_time) to the catalog table
raw_df.write.mode("append").saveAsTable(table_name)

print(f"Loaded raw weather files into {table_name}")
display(spark.table(table_name))


In [0]:
%sql
select * from my.default.raw_weather

In [0]:
from pyspark.sql import functions as F

raw_table = "my.default.raw_weather"
bronze_table = "my.default.bronze_weather"

raw_df = spark.table(raw_table)

# Infer the JSON schema from the first row of file_data
sample_json = raw_df.select("file_data").first()[0]
json_schema = raw_df.select(F.schema_of_json(F.lit(sample_json)).alias("s")).first()["s"]

# Parse the JSON string in file_data into a struct column
parsed_df = raw_df.withColumn("parsed", F.from_json("file_data", json_schema))

# Top-level JSON fields (everything except the nested "hourly" object)
top_level_fields = [f for f in parsed_df.select("parsed.*").schema.fieldNames() if f != "hourly"]

# Hourly variables (everything inside "hourly" except the "time" array)
hourly_vars = [f for f in parsed_df.select("parsed.hourly.*").schema.fieldNames() if f != "time"]

# Explode the hourly time array into one row per timestamp,
# keeping the raw table columns (file_name, ingestion_time, file_data)
# and all top-level JSON fields
exploded_df = parsed_df.select(
    "file_name",
    "ingestion_time",
    "file_data",
    *[F.col(f"parsed.{f}").alias(f) for f in top_level_fields],
    F.col("parsed.hourly").alias("hourly"),
    F.posexplode("parsed.hourly.time").alias("hour_index", "time"),
)

# Align each hourly variable with the exploded timestamp by array position
bronze_df = exploded_df.select(
    "*",
    *[F.expr(f"hourly.{v}[hour_index]").alias(v) for v in hourly_vars],
).drop("hourly", "hour_index")

# Create and insert into the bronze table
bronze_df.write.mode("append").saveAsTable(bronze_table)

print(f"Parsed raw weather records into {bronze_table}")
display(spark.table(bronze_table))


In [0]:
%sql
select count(*) from my.default.bronze_weather

In [0]:
from pyspark.sql import functions as F

bronze_table = "my.default.bronze_weather"
silver_table = "my.default.silver_weather"

# Read the bronze table and add a Fahrenheit column converted from the Celsius temperature
silver_df = (
    spark.table(bronze_table)
    .withColumn(
        "temperature_2m_fahrenheit",
        F.round(F.col("temperature_2m") * 9 / 5 + 32, 2),
    )
)

silver_df.write.mode("append").saveAsTable(silver_table)

print(f"Added Fahrenheit column and loaded into {silver_table}")
display(spark.table(silver_table))

In [0]:
silver_table = "my.default.silver_weather"
gold_table = "my.default.gold_weather"

# Create the gold table (empty, same schema as silver) if it does not exist yet
spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_table} LIKE {silver_table}")

# Merge deduplicated silver data into the gold table: a duplicate is a row whose
# key columns (latitude, longitude, time) match another row. Keep only the most
# recently ingested copy per key (rn = 1) and upsert it into gold.
spark.sql(f"""
MERGE INTO {gold_table} AS t
USING (
  SELECT * EXCEPT (rn)
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY latitude, longitude, time
        ORDER BY ingestion_time DESC
      ) AS rn
    FROM {silver_table}
  )
  WHERE rn = 1
) AS s
ON  t.latitude = s.latitude
AND t.longitude = s.longitude
AND t.time = s.time
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Merged deduplicated data from {silver_table} into {gold_table}")
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {gold_table}"))


# Weather Data Pipeline — Notebook Summary - rajesh

## Overview

This notebook builds a **medallion architecture** (Raw → Bronze → Silver → Gold) pipeline that ingests historical weather data from the **Open-Meteo ERA5 archive API**, stores it in a Unity Catalog Volume, loads it into catalog tables, and progressively refines it into a clean, deduplicated analytics table.

---

## Source Information

| Item | Value |
|---|---|
| API | Open-Meteo ERA5 Historical Weather API |
| Endpoint | `https://archive-api.open-meteo.com/v1/era5` |
| Method | HTTP GET (via `requests`) |
| Parameters | `latitude` (default 52.52), `longitude` (default 13.41), `start_date` (default 2021-01-01), `end_date` (default 2021-12-31), `hourly` variables (default `temperature_2m`) |
| Input mechanism | Databricks widgets — values are configurable at notebook run time |
| Output format | JSON (hourly time series) |

---

## Volume Information

| Item | Value |
|---|---|
| Volume path | `/Volumes/my/default/my` |
| Catalog | `my` |
| Schema | `default` |
| Volume name | `my` |
| Files stored | Raw JSON API responses named `meteo_date_{start_date}_{end_date}.json` |
| Purpose | Landing zone for raw API responses before ingestion into tables |

---

## Medallion / ETL Flow

```
Open-Meteo API (JSON)
        |
        v
Unity Catalog Volume (/Volumes/my/default/my)
        |  spark.read.text (whole-file, pathGlobFilter)
        v
RAW table (my.default.raw_weather)
        |  from_json + posexplode hourly arrays
        v
BRONZE table (my.default.bronze_weather)
        |  temperature_2m_fahrenheit derived column
        v
SILVER table (my.default.silver_weather)
        |  MERGE + ROW_NUMBER dedup (latest ingestion per lat/lon/time)
        v
GOLD table (my.default.gold_weather)
```

### Step-by-Step Operations

1. **Ingest (Extract)** — Widgets collect latitude, longitude, date range, and hourly variables; the API URL is built and fetched with `requests`. The JSON response is saved to the UC Volume.
2. **Raw load (Land)** — Each JSON file in the volume is read as one row (`spark.read.text`, `wholetext=True`, `pathGlobFilter="meteo_date_*.json"`), tagged with `file_name`, `file_data`, and `ingestion_time`, and appended to `raw_weather`.
3. **Bronze (Transform — parse & explode)** — The JSON schema is inferred from a sample row, `file_data` is parsed with `from_json`, the `hourly.time` array is exploded with `posexplode`, and each hourly variable is aligned by array position — producing one row per timestamp. Appended to `bronze_weather`.
4. **Silver (Transform — enrich)** — A `temperature_2m_fahrenheit` column (rounded to 2 decimals) is derived from `temperature_2m` (Celsius). Appended to `silver_weather`.
5. **Gold (Load — dedup & upsert)** — A `MERGE` upserts only the most recently ingested row per key (`latitude`, `longitude`, `time`) into `gold_weather`, keeping the table duplicate-free.

---

## Tables Information

All tables live under catalog **`my`**, schema **`default`**.

| Layer | Table (full path) | Purpose | Key Columns | Write Mode |
|---|---|---|---|---|
| Raw | `my.default.raw_weather` | Unaltered JSON payloads, one row per file | `file_name`, `file_data`, `ingestion_time` | Append |
| Bronze | `my.default.bronze_weather` | Parsed & exploded — one row per hourly timestamp | `file_name`, `ingestion_time`, top-level API fields (e.g. `latitude`, `longitude`), `time`, hourly variables (e.g. `temperature_2m`) | Append |
| Silver | `my.default.silver_weather` | Enriched with Fahrenheit conversion | Bronze columns + `temperature_2m_fahrenheit` | Append |
| Gold | `my.default.gold_weather` | Deduplicated, analytics-ready final table | Silver columns, unique on (`latitude`, `longitude`, `time`) | Merge (upsert) |

---

## Key Takeaways

- **Medallion layers:** Raw (immutable landing) → Bronze (parsed, one row per hour) → Silver (enriched with Fahrenheit) → Gold (deduplicated by latest ingestion time).
- **Idempotent gold:** The MERGE with `ROW_NUMBER() ... ORDER BY ingestion_time DESC` guarantees only the freshest copy of each (`latitude`, `longitude`, `time`) key survives.
- **Fully parameterized:** Changing widget values re-runs the pipeline for any location, date range, or hourly variables without code edits.

